In [1]:
import pandas as pd

In [2]:
# text = pd.read_table(r"C:\Users\AadharJain\Desktop\IBM_Training-main\DAY_6\AI_Sample_text.txt", header=None)
text = '''Artificial intelligence (AI) is intelligence demonstrated by machines, 
as opposed to the natural intelligence displayed by animals including humans. 
AI research has been defined as the field of study of intelligent agents, which refers to any 
system that perceives its environment and takes actions that maximize its chance of achieving 
its goals.

Machine learning is a subset of AI that gives computers the ability to learn without being 
explicitly programmed. It focuses on developing computer programs that can access data and 
use it to learn for themselves. The process begins with observations or data, such as examples, 
direct experience, or instruction, so that computers can look for patterns in data and make 
better decisions in the future.

Deep learning is a subset of machine learning that uses neural networks with many layers. 
These deep neural networks have led to dramatic advances in computer vision, natural language 
processing, and speech recognition. Deep learning models learn representations of data with 
multiple levels of abstraction.

Natural language processing (NLP) is a subfield of linguistics, computer science, and 
artificial intelligence concerned with the interactions between computers and human language, 
in particular how to program computers to process and analyze large amounts of natural 
language data. The goal is a computer capable of understanding the contents of documents, 
including the contextual nuances of the language within them.

Retrieval Augmented Generation (RAG) is a technique that combines retrieval systems with 
generative models. Instead of relying solely on the parametric knowledge stored in model 
weights, RAG systems retrieve relevant documents from an external knowledge base and use 
them as context when generating responses. This allows models to access up-to-date information 
and reduces hallucination.'''

In [3]:
def fixed_size_chunk(text:str, chunk_size, overlap):
    chunks = []
    start = 0
    chunk_id = 0

    while start < len(text):
        end = min(start+chunk_size, len(text))
        chunk_text = text[start:end]
        chunks.append({
            'id': chunk_id,
            'text': chunk_text,
            'start': start,
            'end': end,
            'size': len(chunk_text)
        })

        print(chunks[chunk_id])
        chunk_id += 1
        start = end-overlap

        # if start >= len(text):
        #     break

        if end >= len(text)-1:
            break

    return chunks

In [4]:
chunks = fixed_size_chunk(text, 300,  50)

{'id': 0, 'text': 'Artificial intelligence (AI) is intelligence demonstrated by machines, \nas opposed to the natural intelligence displayed by animals including humans. \nAI research has been defined as the field of study of intelligent agents, which refers to any \nsystem that perceives its environment and takes action', 'start': 0, 'end': 300, 'size': 300}
{'id': 1, 'text': 'em that perceives its environment and takes actions that maximize its chance of achieving \nits goals.\n\nMachine learning is a subset of AI that gives computers the ability to learn without being \nexplicitly programmed. It focuses on developing computer programs that can access data and \nuse it to lea', 'start': 250, 'end': 550, 'size': 300}
{'id': 2, 'text': 'r programs that can access data and \nuse it to learn for themselves. The process begins with observations or data, such as examples, \ndirect experience, or instruction, so that computers can look for patterns in data and make \nbetter decisions in the

In [5]:
from nltk import sent_tokenize

In [6]:
def sentence_chunking(text: str, sentences_per_chunk, overlap_sentences):
    sentences = sent_tokenize(text)
    # sentences = simple_sent_tokenize(text)

    chunks = []
    step = max(1, sentences_per_chunk - overlap_sentences)

    for i in range(0, len(sentences), step):
        chunks_sents = sentences[i : i + sentences_per_chunk]
        if chunks_sents:
            chunks.append({
                'id': len(chunks),
                'text': ' '.join(chunks_sents),
                'sentence_range': (i, i + len(chunks_sents)-1),
                'size': len(' '.join(chunks_sents)),
                'num_sentences': len(chunks_sents)
            })

            # print(chunks[len(chunks)-1])

    return chunks

In [7]:
chunks = sentence_chunking(text, 3, 1)

**Paragraph Chunking**

In [8]:
import re

In [9]:
def paragraph_chunk(text: str, min_chars, max_chars):
    paragraphs = re.split(r'\n\n+', text)
    chunks = []

    for para in paragraphs:
        para = re.sub(r'\s+',' ',para).strip()

        if len(para) < min_chars:
            continue

        if max_chars and len(para) > max_chars:
            sub_chunks = sentence_chunking(para, sentences_per_chunk=3, overlap_sentences=1)
            for sc in sub_chunks:
                chunks.append({
                    'id': len(chunks),
                    'text': sc['text'],
                    'size': sc['size'],
                    'source': 'sentence_chunking'
                })

        else:
            chunks.append({
                'id': len(chunks),
                'text': para,
                'size': len(para),
                'source': 'para_chunking'
            })
                        
    return chunks

In [10]:
para_chunks = paragraph_chunk(text, 50, 100)
for i in range(len(para_chunks)):
    print(para_chunks[i])

{'id': 0, 'text': 'Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the natural intelligence displayed by animals including humans. AI research has been defined as the field of study of intelligent agents, which refers to any system that perceives its environment and takes actions that maximize its chance of achieving its goals.', 'size': 347, 'source': 'sentence_chunking'}
{'id': 1, 'text': 'Machine learning is a subset of AI that gives computers the ability to learn without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves. The process begins with observations or data, such as examples, direct experience, or instruction, so that computers can look for patterns in data and make better decisions in the future.', 'size': 401, 'source': 'sentence_chunking'}
{'id': 2, 'text': 'The process begins with observations or data, such as examples, direct experience, or instruction, so

**bow_Tf_idf_implementation**

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

In [62]:
paragraphs = []

# getting chunks out into the paragraphs
for para in para_chunks:
    paragraphs.append( para['text'] )

user_query = input("Enter a query to look up: ")
all_texts = paragraphs + [user_query]

In [50]:
bow_vectorizer = CountVectorizer(stop_words='english')
bow_matrix = bow_vectorizer.fit_transform(all_texts)

query_bow_vector = bow_matrix[-1]
paragraph_bow_vector = bow_matrix[:-1]


In [51]:
from sklearn.metrics.pairwise import cosine_similarity
bow_similarity = cosine_similarity(query_bow_vector, paragraph_bow_vector)
bow_similarity.shape

(1, 8)

In [52]:
ls = list(bow_similarity[0])
print(para_chunks[ls.index(max(ls))]['text'])

Deep learning is a subset of machine learning that uses neural networks with many layers. These deep neural networks have led to dramatic advances in computer vision, natural language processing, and speech recognition. Deep learning models learn representations of data with multiple levels of abstraction.


**Tf-IDF**

In [53]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(all_texts)

query_tfidf_vector = tfidf_matrix[-1]
paragraph_tfidf_vector = tfidf_matrix[:-1]

In [54]:
from sklearn.metrics.pairwise import cosine_similarity
tfidf_similarity = cosine_similarity(query_tfidf_vector, paragraph_tfidf_vector)
tfidf_similarity.shape

(1, 8)

In [55]:
ls = list(tfidf_similarity[0])
print(para_chunks[ls.index(max(ls))]['text'])

Deep learning is a subset of machine learning that uses neural networks with many layers. These deep neural networks have led to dramatic advances in computer vision, natural language processing, and speech recognition. Deep learning models learn representations of data with multiple levels of abstraction.


**pre trained model**

In [59]:
from sentence_transformers import SentenceTransformer
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\AadharJain\Desktop\IBM_Training-main\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\AadharJain\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5800.24it/s]


In [60]:
paragraph_embeddings = sentence_model.encode(paragraphs)
query_embeddings = sentence_model.encode([user_query])
similarity = cosine_similarity(query_embeddings, paragraph_embeddings)
similarity.shape


(1, 8)

In [61]:
ls = list(similarity[0])
print(para_chunks[ls.index(max(ls))]['text'])

Deep learning is a subset of machine learning that uses neural networks with many layers. These deep neural networks have led to dramatic advances in computer vision, natural language processing, and speech recognition. Deep learning models learn representations of data with multiple levels of abstraction.
